# HydroSense-Kenya: Smart Irrigation System
## Level 3 - Core Numerical Methods Engine

Student: ELPIS MWANGI MAINA
Student Number: SCT211-0003/2024

## Layer 3 Overview

This layer implements core numerical methods from scratch and applies them to 
irrigation decision-making. Rather than calling library functions, we build 
each algorithm manually to understand how they work.

Main objectives:
    Implement root finding (bisection, Newton-Raphson, secant)
    Estimate rates of change with numerical differentiation
    Compute integrals with numerical integration
    Solve linear systems with Gaussian elimination
    Apply all methods to irrigation optimization problems

In [20]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numerical_methods import RootFinding, NumericalDifferentiation, NumericalIntegration, LinearSystems, IrrigationOptimization

# Load data
weather = pd.read_csv('../data/raw/weather_daily.csv')
soil = pd.read_csv('../data/raw/soil_sensor_data.csv')
params = pd.read_csv('../data/raw/crop_zone_parameters.csv')

weather['date'] = pd.to_datetime(weather['date'])
soil['timestamp'] = pd.to_datetime(soil['timestamp'])

print("Numerical methods module loaded")
print(f"Weather data: {len(weather)} days")
print("Ready for analysis")


Numerical methods module loaded
Weather data: 30 days
Ready for analysis


## Concept 1: Root Finding Methods

Find the irrigation amount needed to reach target soil moisture using three methods:
bisection, Newton-Raphson, and secant. Each method has different convergence 
characteristics and requirements.

Problem: Calculate irrigation I such that soil moisture reaches 35% given:
    Current moisture: 28%
    Rainfall: 5mm
    ET: 2.5mm
    Field capacity: 45%
    Drainage coefficient: 0.18

In [21]:
# Define the water balance equation
def water_balance(irrigation, S_current=28, rainfall=5, ET=2.5, FC=45, drain_coeff=0.18):
    available = S_current + rainfall + irrigation
    drainage = drain_coeff * max(0, available - FC)
    S_next = available - ET - drainage
    return S_next

# We want to find I such that S_next = 35
def target_equation(irrigation):
    return water_balance(irrigation) - 35.0

# And the derivative for Newton-Raphson
def target_derivative(irrigation, h=1e-5):
    return (target_equation(irrigation + h) - target_equation(irrigation)) / h

# Test the three root finding methods
print("Root Finding Methods for Irrigation Calculation")
print("=" * 60)

# Bisection method
bisection_result = RootFinding.bisection(target_equation, 0, 50)
print(f"\nBisection method:")
print(f"  Irrigation required: {bisection_result['root']:.4f} mm")
print(f"  Iterations: {bisection_result['iterations']}")
print(f"  Error: {bisection_result['error']:.2e}")
print(f"  Converged: {bisection_result['converged']}")

# Newton-Raphson method
nr_result = RootFinding.newton_raphson(target_equation, target_derivative, x0=10)
print(f"\nNewton-Raphson method:")
print(f"  Irrigation required: {nr_result['root']:.4f} mm")
print(f"  Iterations: {nr_result['iterations']}")
print(f"  Error: {nr_result['error']:.2e}")
print(f"  Converged: {nr_result['converged']}")

# Secant method
secant_result = RootFinding.secant(target_equation, 0, 20)
print(f"\nSecant method:")
print(f"  Irrigation required: {secant_result['root']:.4f} mm")
print(f"  Iterations: {secant_result['iterations']}")
print(f"  Error: {secant_result['error']:.2e}")
print(f"  Converged: {secant_result['converged']}")


Root Finding Methods for Irrigation Calculation

Bisection method:
  Irrigation required: 4.5000 mm
  Iterations: 26
  Error: 7.45e-07
  Converged: True

Newton-Raphson method:
  Irrigation required: 4.5000 mm
  Iterations: 2
  Error: 1.75e-09
  Converged: True

Secant method:
  Irrigation required: 4.5000 mm
  Iterations: 4
  Error: 0.00e+00
  Converged: True


## Concept 2: Numerical Differentiation

Estimate the rate of soil moisture change using forward, backward, and central 
differences. This helps understand how sensitive moisture is to changes in inputs.

In [22]:
# Define soil moisture as a function of rainfall
def soil_moisture_from_rainfall(rainfall, S_current=28, ET=2.5, FC=45, drain_coeff=0.18):
    available = S_current + rainfall
    drainage = drain_coeff * max(0, available - FC)
    S_next = available - ET - drainage
    return S_next

# Calculate derivative at rainfall = 5mm
rainfall_point = 5.0

d_forward = NumericalDifferentiation.forward_difference(soil_moisture_from_rainfall, rainfall_point)
d_backward = NumericalDifferentiation.backward_difference(soil_moisture_from_rainfall, rainfall_point)
d_central = NumericalDifferentiation.central_difference(soil_moisture_from_rainfall, rainfall_point)

print("Numerical Differentiation: dS/dRainfall at rainfall=5mm")
print("=" * 60)
print(f"Forward difference:  {d_forward:.6f} %/mm")
print(f"Backward difference: {d_backward:.6f} %/mm")
print(f"Central difference:  {d_central:.6f} %/mm")
print(f"\nInterpretation: Each 1mm of rainfall increases soil moisture by ~{d_central:.3f}%")


Numerical Differentiation: dS/dRainfall at rainfall=5mm
Forward difference:  1.000000 %/mm
Backward difference: 1.000000 %/mm
Central difference:  1.000000 %/mm

Interpretation: Each 1mm of rainfall increases soil moisture by ~1.000%


## Concept 3: Numerical Integration

Estimate cumulative water deficit over 30 days using trapezoidal and Simpson's rules.
This helps understand total water needs across the growing period.

In [23]:
# Calculate daily ET requirement
def ET_requirement(day):
    idx = min(int(day), len(weather) - 1)
    row = weather.iloc[idx]
    ET = max(0, 0.12*row['temperature_c'] + 0.35*row['wind_speed_mps'] + 
             2.4*row['solar_index'] - 0.025*row['humidity_pct'])
    return ET

# Integrate ET over 30 days
integral_trap = NumericalIntegration.trapezoidal(ET_requirement, 0, 29, n=30)
integral_simp = NumericalIntegration.simpson(ET_requirement, 0, 29, n=30)

print("Numerical Integration: Cumulative Water Requirement")
print("=" * 60)
print(f"Trapezoidal rule: {integral_trap:.2f} mm")
print(f"Simpson's rule:   {integral_simp:.2f} mm")
print(f"Difference:       {abs(integral_trap - integral_simp):.3f} mm")
print(f"\nInterpretation: Total ET over 30 days is approximately {integral_simp:.1f} mm")


Numerical Integration: Cumulative Water Requirement
Trapezoidal rule: 104.78 mm
Simpson's rule:   102.66 mm
Difference:       2.128 mm

Interpretation: Total ET over 30 days is approximately 102.7 mm


## Concept 4: Linear Systems - Water Allocation Problem

Solve a three-zone water allocation problem using Gaussian elimination.
Objective: Distribute 500mm of water among three zones to meet their demand.

In [24]:
# Three-zone water allocation problem Ax = b
# Zone A (Tomato), Zone B (Kale), Zone C (Maize)
# Each zone has a water requirement and area constraint

# Coefficient matrix A
A = np.array([
    [1.0, 1.0, 1.0],        # Total water: x_A + x_B + x_C = 500
    [2.0, 1.5, 1.0],        # Water efficiency: 2*x_A + 1.5*x_B + 1.0*x_C = 800
    [0.5, 2.0, 1.5]         # Yield priority: 0.5*x_A + 2*x_B + 1.5*x_C = 650
])

# Right-hand side b
b = np.array([500.0, 800.0, 650.0])

# Solve using Gaussian elimination
x = LinearSystems.gaussian_elimination(A, b)

print("Three-Zone Water Allocation Problem")
print("=" * 60)
print("System Ax = b:")
print(f"  Equation 1 (Total water):    x_A + x_B + x_C = 500")
print(f"  Equation 2 (Efficiency):    2*x_A + 1.5*x_B + 1.0*x_C = 800")
print(f"  Equation 3 (Yield priority): 1*x_A + 2*x_B + 3*x_C = 1100")
print()
print("Solution:")
print(f"  Zone A (Tomato): {x[0]:.2f} mm")
print(f"  Zone B (Kale):   {x[1]:.2f} mm")
print(f"  Zone C (Maize):  {x[2]:.2f} mm")
print(f"  Total:           {np.sum(x):.2f} mm")

# Verify solution
residual = A @ x - b
print(f"\nVerification (should be near zero):")
print(f"  Residual: {np.linalg.norm(residual):.2e}")


Three-Zone Water Allocation Problem
System Ax = b:
  Equation 1 (Total water):    x_A + x_B + x_C = 500
  Equation 2 (Efficiency):    2*x_A + 1.5*x_B + 1.0*x_C = 800
  Equation 3 (Yield priority): 1*x_A + 2*x_B + 3*x_C = 1100

Solution:
  Zone A (Tomato): 200.00 mm
  Zone B (Kale):   200.00 mm
  Zone C (Maize):  100.00 mm
  Total:           500.00 mm

Verification (should be near zero):
  Residual: 5.68e-14


## Layer 3 Summary

Layer 3 demonstrates core numerical methods:

Root Finding:
    Bisection: Reliable, slower, always converges
    Newton-Raphson: Faster, needs derivative, may not converge
    Secant: Fast, no derivative needed, may not converge
    Application: Calculate irrigation amounts for target moisture

Numerical Differentiation:
    Approximates derivatives numerically
    Central difference most accurate
    Application: Understand sensitivity of moisture to inputs

Numerical Integration:
    Trapezoidal rule: Simple, less accurate
    Simpson's rule: More accurate with parabolic approximations
    Application: Estimate cumulative water requirements

Linear Systems:
    Gaussian elimination: Direct method with pivoting
    Application: Solve multi-zone water allocation problems

All methods implemented from first principles to demonstrate 
understanding of numerical computation.